# Фінальний проєкт. Relational Databases for DS

**Датасет:** `infectious_cases.csv` (статистика інфекційних захворювань за країнами та роками).

## 1. Завантаження даних

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/fp_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://', 1), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

def sql(q):
    with engine.begin() as conn:
        cur = conn.connection.cursor()
        cur.execute('SET search_path TO pandemic, public;')
        cur.execute(q)
        if cur.description:
            return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])

PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit


In [3]:
from urllib.request import urlretrieve

csv_path = 'data/infectious_cases.csv'
if not os.path.exists(csv_path):
    os.makedirs('data', exist_ok=True)
    urlretrieve('https://drive.google.com/uc?export=download&id=1dQf-uVXzYPYYGgXsG1y4Q1qDdhP2miZl', csv_path)
print(os.path.getsize(csv_path))

540895


In [4]:
sql("""
CREATE SCHEMA IF NOT EXISTS pandemic;
SET search_path TO pandemic;
""")

In [5]:
sql("""
DROP TABLE IF EXISTS infectious_cases_norm CASCADE;
DROP TABLE IF EXISTS entities CASCADE;
DROP TABLE IF EXISTS infectious_cases CASCADE;

CREATE TABLE infectious_cases (
    entity TEXT,
    code TEXT,
    year TEXT,
    number_yaws TEXT,
    polio_cases TEXT,
    cases_guinea_worm TEXT,
    number_rabies TEXT,
    number_malaria TEXT,
    number_hiv TEXT,
    number_tuberculosis TEXT,
    number_smallpox TEXT,
    number_cholera_cases TEXT
);
""")

In [6]:
raw_conn = engine.raw_connection()
try:
    with raw_conn.cursor() as cur:
        with open(csv_path, 'r', encoding='utf-8', newline='') as file:
            cur.copy_expert(
                "COPY pandemic.infectious_cases FROM STDIN WITH (FORMAT csv, HEADER true, DELIMITER ',')",
                file
            )
    raw_conn.commit()
finally:
    raw_conn.close()

In [7]:
sql("""
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT entity) AS n_entities,
       MIN(year::INTEGER) AS min_year,
       MAX(year::INTEGER) AS max_year,
       COUNT(*) FILTER (WHERE number_rabies IS NULL) AS rabies_missing
FROM infectious_cases;
""")

,n_rows,n_entities,min_year,max_year,rabies_missing
0,10521,245,1920,2023,3741


In [8]:
sql("""
SELECT *
FROM infectious_cases
LIMIT 5;
""")

,entity,code,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,Afghanistan,AFG,1980,None,6160,0,None,None,None,None,None,None
1,Afghanistan,AFG,1981,None,5859,0,None,None,None,None,None,None
2,Afghanistan,AFG,1982,None,9730,0,None,None,None,None,None,None
3,Afghanistan,AFG,1983,None,13937,0,None,None,None,None,None,None
4,Afghanistan,AFG,1984,None,3864,0,None,None,None,None,None,None


Створено схему `pandemic`, її встановлено як `search_path`, а CSV імпортовано через `COPY FROM STDIN` у таблицю `infectious_cases`. Таблиця містить 10 521 рядок для 245 сутностей за 1920–2023 роки. Порожні клітинки CSV `COPY` завантажує як `NULL`; для `number_rabies` таких рядків 3 741.

## 2. Нормалізація до 3НФ

**Функціональні залежності:** `entity → code` (кожна сутність має один код), а показники захворювань залежать від пари `(entity, year)`. У вихідній таблиці `entity` і `code` повторюються в кожному рядку, а `code` транзитивно залежить від `entity`, тому таблиця не в 3НФ. Після декомпозиції `entities` зберігає кожну пару `entity`/`code` один раз, а `infectious_cases_norm` посилається на неї через `entity_id` (FK) з ключем `(entity_id, year)`. Для регіонів без коду (`Africa`, `World (WB)` тощо) `code` зберігається як `NULL`, тому для унікальності використано `UNIQUE NULLS NOT DISTINCT`.

In [9]:
sql("""
CREATE TABLE entities (
    entity_id SERIAL PRIMARY KEY,
    entity TEXT NOT NULL,
    code TEXT,
    UNIQUE NULLS NOT DISTINCT (entity, code)
);

INSERT INTO entities (entity, code)
SELECT DISTINCT entity, NULLIF(code, '')
FROM infectious_cases
ORDER BY entity;

CREATE TABLE infectious_cases_norm (
    case_id SERIAL PRIMARY KEY,
    entity_id INTEGER NOT NULL REFERENCES entities(entity_id),
    year SMALLINT NOT NULL,
    number_yaws NUMERIC,
    polio_cases NUMERIC,
    cases_guinea_worm NUMERIC,
    number_rabies NUMERIC,
    number_malaria NUMERIC,
    number_hiv NUMERIC,
    number_tuberculosis NUMERIC,
    number_smallpox NUMERIC,
    number_cholera_cases NUMERIC,
    UNIQUE (entity_id, year)
);

INSERT INTO infectious_cases_norm (
    entity_id, year, number_yaws, polio_cases, cases_guinea_worm, number_rabies,
    number_malaria, number_hiv, number_tuberculosis, number_smallpox, number_cholera_cases
)
SELECT
    e.entity_id,
    ic.year::SMALLINT,
    NULLIF(ic.number_yaws, '')::NUMERIC,
    NULLIF(ic.polio_cases, '')::NUMERIC,
    NULLIF(ic.cases_guinea_worm, '')::NUMERIC,
    NULLIF(ic.number_rabies, '')::NUMERIC,
    NULLIF(ic.number_malaria, '')::NUMERIC,
    NULLIF(ic.number_hiv, '')::NUMERIC,
    NULLIF(ic.number_tuberculosis, '')::NUMERIC,
    NULLIF(ic.number_smallpox, '')::NUMERIC,
    NULLIF(ic.number_cholera_cases, '')::NUMERIC
FROM infectious_cases AS ic
JOIN entities AS e
  ON e.entity = ic.entity
 AND e.code IS NOT DISTINCT FROM NULLIF(ic.code, '');
""")

In [10]:
sql("""
SELECT 'infectious_cases' AS table_name, COUNT(*) AS n_rows FROM infectious_cases
UNION ALL SELECT 'entities', COUNT(*) FROM entities
UNION ALL SELECT 'infectious_cases_norm', COUNT(*) FROM infectious_cases_norm;
""")

,table_name,n_rows
0,infectious_cases,10521
1,entities,245
2,infectious_cases_norm,10521


In [11]:
sql("""
SELECT e.entity_id, e.entity, e.code, n.year, n.number_rabies, n.number_malaria
FROM infectious_cases_norm AS n
JOIN entities AS e ON e.entity_id = n.entity_id
ORDER BY e.entity, n.year
LIMIT 5;
""")

,entity_id,entity,code,year,number_rabies,number_malaria
0,1,Afghanistan,AFG,1960,None,None
1,1,Afghanistan,AFG,1965,None,None
2,1,Afghanistan,AFG,1980,None,None
3,1,Afghanistan,AFG,1981,None,None
4,1,Afghanistan,AFG,1982,None,None


Обидві нормалізовані таблиці збережено в схемі `pandemic`: `entities` — 245 рядків, `infectious_cases_norm` — 10 521 рядок, тобто кожен вихідний рядок отримав свій `entity_id`. Числові показники приведено з TEXT до `NUMERIC`.

## 3. Аналіз Number_rabies

In [12]:
sql("""
SELECT
    e.entity_id,
    e.entity,
    e.code,
    ROUND(AVG(n.number_rabies), 2) AS avg_rabies,
    ROUND(MIN(n.number_rabies), 2) AS min_rabies,
    ROUND(MAX(n.number_rabies), 2) AS max_rabies,
    ROUND(SUM(n.number_rabies), 2) AS sum_rabies
FROM infectious_cases_norm AS n
JOIN entities AS e ON e.entity_id = n.entity_id
WHERE n.number_rabies IS NOT NULL
GROUP BY e.entity_id, e.entity, e.code
ORDER BY AVG(n.number_rabies) DESC
LIMIT 10;
""")

,entity_id,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies
0,241,World,OWID_WRL,20192.37,14075.51,24744.66,605771.11
1,121,Lower Middle Income (WB),NaN,15193.96,10202.53,19182.80,455818.78
2,202,South Asia (WB),NaN,11729.89,7271.28,15361.88,351896.66
3,206,South-East Asia Region (WHO),NaN,11424.33,6806.01,15641.96,342729.81
4,77,G20,NaN,10189.05,6339.08,13164.88,305671.38
5,97,India,IND,8599.17,5425.87,11121.14,257975.20
6,209,Sub-Saharan Africa (WB),NaN,4748.27,4356.82,5017.98,142448.04
7,3,African Region (WHO),NaN,4649.70,4283.07,4917.78,139491.07
8,120,Low Income (WB),NaN,3568.11,3014.62,3873.53,107043.16
9,58,East Asia & Pacific (WB),NaN,3427.46,2009.94,4591.01,102823.83


Агрегати пораховано лише для рядків із числовим `number_rabies` (`IS NOT NULL`), відсортовано за середнім за спаданням і обмежено 10 рядками. Перше місце займає агрегат `World` (≈20 192 у середньому на рік), а серед окремих країн у топ-10 потрапила лише `India` (≈8 599). Решта — регіональні та income-групи, тому для порівняння країн агрегати варто відфільтрувати за `code IS NOT NULL`.

## 4. Колонка різниці в роках

In [13]:
sql("""
SELECT
    year,
    MAKE_DATE(year, 1, 1) AS year_start_date,
    CURRENT_DATE AS today,
    EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(year, 1, 1)))::INTEGER AS years_diff
FROM infectious_cases_norm
ORDER BY entity_id, year
LIMIT 10;
""")

,year,year_start_date,today,years_diff
0,1960,1960-01-01,2026-10-14,66
1,1965,1965-01-01,2026-10-14,61
2,1980,1980-01-01,2026-10-14,46
3,1981,1981-01-01,2026-10-14,45
4,1982,1982-01-01,2026-10-14,44
5,1983,1983-01-01,2026-10-14,43
6,1984,1984-01-01,2026-10-14,42
7,1985,1985-01-01,2026-10-14,41
8,1986,1986-01-01,2026-10-14,40
9,1987,1987-01-01,2026-10-14,39


`MAKE_DATE(year, 1, 1)` створює дату 1 січня відповідного року, `CURRENT_DATE` повертає поточну дату, а `EXTRACT(YEAR FROM AGE(...))` — кількість повних років між ними.

## 5. Власні функції

In [14]:
sql("""
CREATE OR REPLACE FUNCTION years_since(p_year INTEGER)
RETURNS INTEGER
LANGUAGE SQL
STABLE
AS $$
    SELECT EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(p_year, 1, 1)))::INTEGER;
$$;

CREATE OR REPLACE FUNCTION avg_cases_per_period(p_cases NUMERIC, p_divider INTEGER)
RETURNS NUMERIC
LANGUAGE SQL
IMMUTABLE
AS $$
    SELECT p_cases / NULLIF(p_divider, 0);
$$;
""")

In [15]:
sql("""
SELECT
    year,
    MAKE_DATE(year, 1, 1) AS year_start_date,
    years_since(year) AS years_diff
FROM infectious_cases_norm
ORDER BY entity_id, year
LIMIT 10;
""")

,year,year_start_date,years_diff
0,1960,1960-01-01,66
1,1965,1965-01-01,61
2,1980,1980-01-01,46
3,1981,1981-01-01,45
4,1982,1982-01-01,44
5,1983,1983-01-01,43
6,1984,1984-01-01,42
7,1985,1985-01-01,41
8,1986,1986-01-01,40
9,1987,1987-01-01,39


In [16]:
sql("""
SELECT
    e.entity,
    n.year,
    n.number_rabies,
    ROUND(avg_cases_per_period(n.number_rabies, 12), 4) AS avg_per_month,
    ROUND(avg_cases_per_period(n.number_rabies, 4), 4) AS avg_per_quarter,
    ROUND(avg_cases_per_period(n.number_rabies, 2), 4) AS avg_per_half_year
FROM infectious_cases_norm AS n
JOIN entities AS e ON e.entity_id = n.entity_id
WHERE n.number_rabies IS NOT NULL
ORDER BY n.number_rabies DESC
LIMIT 10;
""")

,entity,year,number_rabies,avg_per_month,avg_per_quarter,avg_per_half_year
0,World,1990,24744.658,2062.0548,6186.1645,12372.3290
1,World,1991,24559.182,2046.5985,6139.7955,12279.5910
2,World,1992,24337.137,2028.0948,6084.2843,12168.5685
3,World,1993,24127.453,2010.6211,6031.8633,12063.7265
4,World,1994,23972.55,1997.7125,5993.1375,11986.2750
5,World,1995,23905.242,1992.1035,5976.3105,11952.6210
6,World,1996,23831.625,1985.9688,5957.9063,11915.8125
7,World,1997,23652.562,1971.0468,5913.1405,11826.2810
8,World,1998,23415.79,1951.3158,5853.9475,11707.8950
9,World,1999,23185.303,1932.1086,5796.3258,11592.6515


`years_since(year)` повертає той самий результат, що й колонка з завдання 4. `avg_cases_per_period(cases, divider)` ділить річну кількість захворювань на дільник (12 — місяць, 4 — квартал, 2 — півріччя); `NULLIF` захищає від ділення на 0. Функцію застосовано лише до рядків, де `number_rabies` має числове значення.